In [41]:
import json
from pathlib import Path

# Change this if your notebook has a different filename
notebook_path = Path("02_sql_feature_engineering.ipynb")

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

for i, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] == "code":
        source = "".join(cell["source"]).strip()
        first_line = source.split("\n")[0] if source else "(empty cell)"
        print(f"Cell {i}: {first_line[:120]}")

Cell 0: import json
Cell 1: import duckdb
Cell 2: query = f"""
Cell 3: query = f"""
Cell 4: query = f"""
Cell 5: query = f"""
Cell 6: processed_dir = DATA_DIR.parent / "processed"
Cell 7: import pandas as pd
Cell 8: train_model = train.merge(
Cell 9: %pip install lightgbm
Cell 10: from sklearn.model_selection import train_test_split
Cell 11: # E2 — APPLICATION + PREVIOUS APPLICATIONS
Cell 12: preprocessor_2 = ColumnTransformer(
Cell 13: print("ROC-AUC improvement:",
Cell 14: from lightgbm import LGBMClassifier
Cell 15: bureau_query = f"""
Cell 16: bureau_features.to_parquet(
Cell 17: train_model_e4 = train_model.merge(
Cell 18: # Additional bureau risk ratios
Cell 19: X_e4 = train_model_e4.drop(
Cell 20: lgbm_e4 = LGBMClassifier(
Cell 21: installments_query = """
Cell 22: installment_features.to_parquet(
Cell 23: train_model_e5 = train_model_e4.merge(
Cell 24: X_e5 = train_model_e5.drop(
Cell 25: lgbm_e5 = LGBMClassifier(
Cell 26: pos_query = """
Cell 27: pos_features.to_parquet(
Cell 

In [3]:
import duckdb
from pathlib import Path

DATA_DIR = Path("../data/raw")

con = duckdb.connect()

print("DuckDB ready.")

DuckDB ready.


In [4]:
query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT SK_ID_CURR) AS unique_applicants
FROM read_csv_auto('{DATA_DIR / "previous_application.csv"}')
"""

con.sql(query)

┌────────────┬───────────────────┐
│ total_rows │ unique_applicants │
│   int64    │       int64       │
├────────────┼───────────────────┤
│    1670214 │            338857 │
└────────────┴───────────────────┘

In [5]:
query = f"""
SELECT
    NAME_CONTRACT_STATUS,
    COUNT(*) AS applications
FROM read_csv_auto('{DATA_DIR / "previous_application.csv"}')
GROUP BY NAME_CONTRACT_STATUS
ORDER BY applications DESC
"""

con.sql(query)

┌──────────────────────┬──────────────┐
│ NAME_CONTRACT_STATUS │ applications │
│       varchar        │    int64     │
├──────────────────────┼──────────────┤
│ Approved             │      1036781 │
│ Canceled             │       316319 │
│ Refused              │       290678 │
│ Unused offer         │        26436 │
└──────────────────────┴──────────────┘

In [6]:
query = f"""
SELECT
    CREDIT_ACTIVE,
    COUNT(*) AS records
FROM read_csv_auto('{DATA_DIR / "bureau.csv"}')
GROUP BY CREDIT_ACTIVE
ORDER BY records DESC
"""

con.sql(query)

┌───────────────┬─────────┐
│ CREDIT_ACTIVE │ records │
│    varchar    │  int64  │
├───────────────┼─────────┤
│ Closed        │ 1079273 │
│ Active        │  630607 │
│ Sold          │    6527 │
│ Bad debt      │      21 │
└───────────────┴─────────┘

In [7]:
query = f"""
SELECT
    SK_ID_CURR,

    -- Volume
    COUNT(*) AS prev_app_count,

    -- Application outcomes
    SUM(CASE
        WHEN NAME_CONTRACT_STATUS = 'Approved' THEN 1
        ELSE 0
    END) AS prev_approved_count,

    SUM(CASE
        WHEN NAME_CONTRACT_STATUS = 'Refused' THEN 1
        ELSE 0
    END) AS prev_refused_count,

    SUM(CASE
        WHEN NAME_CONTRACT_STATUS = 'Canceled' THEN 1
        ELSE 0
    END) AS prev_canceled_count,

    -- Approval / refusal behavior
    AVG(CASE
        WHEN NAME_CONTRACT_STATUS = 'Approved' THEN 1.0
        ELSE 0.0
    END) AS prev_approval_rate,

    AVG(CASE
        WHEN NAME_CONTRACT_STATUS = 'Refused' THEN 1.0
        ELSE 0.0
    END) AS prev_refusal_rate,

    -- Requested / approved credit behavior
    AVG(AMT_APPLICATION) AS prev_avg_application,

    AVG(AMT_CREDIT) AS prev_avg_credit,

    MAX(AMT_CREDIT) AS prev_max_credit,

    -- Down payment behavior
    AVG(AMT_DOWN_PAYMENT) AS prev_avg_down_payment,

    -- Credit utilization of requested amount
    AVG(
        CASE
            WHEN AMT_APPLICATION > 0
            THEN AMT_CREDIT / AMT_APPLICATION
            ELSE NULL
        END
    ) AS prev_avg_credit_application_ratio,

    -- Loan duration
    AVG(CNT_PAYMENT) AS prev_avg_installments,

    -- Decision timing
    AVG(DAYS_DECISION) AS prev_avg_days_decision,

    MIN(DAYS_DECISION) AS prev_most_recent_decision_days,

    -- Product mix
    SUM(CASE
        WHEN NAME_CONTRACT_TYPE = 'Cash loans' THEN 1
        ELSE 0
    END) AS prev_cash_loan_count,

    SUM(CASE
        WHEN NAME_CONTRACT_TYPE = 'Revolving loans' THEN 1
        ELSE 0
    END) AS prev_revolving_loan_count

FROM read_csv_auto(
    '{DATA_DIR / "previous_application.csv"}'
)

GROUP BY SK_ID_CURR
"""

previous_features = con.sql(query).df()

print("Feature table shape:", previous_features.shape)

previous_features.head()

Feature table shape: (338857, 17)


,SK_ID_CURR,prev_app_count,prev_approved_count,prev_refused_count,prev_canceled_count,prev_approval_rate,prev_refusal_rate,prev_avg_application,prev_avg_credit,prev_max_credit,prev_avg_down_payment,prev_avg_credit_application_ratio,prev_avg_installments,prev_avg_days_decision,prev_most_recent_decision_days,prev_cash_loan_count,prev_revolving_loan_count
0,142223,21,5.0,7.0,9.0,0.238095,0.333333,185227.928571,220027.5000,779796.0,0.00,1.173896,28.00,-574.761905,-1233,20.0,0.0
1,218295,8,5.0,1.0,2.0,0.625000,0.125000,703687.500000,797561.4375,1408806.0,NaN,1.053328,15.00,-602.375000,-1536,7.0,1.0
2,111074,18,8.0,4.0,6.0,0.444444,0.222222,370192.250000,432682.2500,1661418.0,2899.50,1.015326,32.25,-533.500000,-2420,14.0,1.0
3,268730,1,1.0,0.0,0.0,1.000000,0.000000,33426.000000,30082.5000,30082.5,3343.50,0.899973,12.00,-406.000000,-406,0.0,0.0
4,186356,10,2.0,5.0,3.0,0.200000,0.500000,100584.000000,114261.3000,843552.0,15131.25,1.032403,21.00,-347.000000,-510,7.0,1.0


In [42]:
from pathlib import Path
import pandas as pd

processed_dir = DATA_DIR.parent / "processed"
processed_dir.mkdir(exist_ok=True)

previous_features_path = (
    processed_dir / "previous_application_features.parquet"
)

if previous_features_path.exists():
    previous_features = pd.read_parquet(previous_features_path)
    print("Loaded saved previous-application features.")
else:
    # This assumes the earlier feature-engineering cell
    # has already created previous_features.
    previous_features.to_parquet(
        previous_features_path,
        index=False
    )
    print("Saved newly generated previous-application features.")

print("Shape:", previous_features.shape)
print("Unique applicants:", previous_features["SK_ID_CURR"].nunique())
print("Duplicate applicant IDs:", previous_features["SK_ID_CURR"].duplicated().sum())

Loaded saved previous-application features.
Shape: (338857, 17)
Unique applicants: 338857
Duplicate applicant IDs: 0


In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/raw")

# Load the application datasets into this notebook
train = pd.read_csv(DATA_DIR / "application_train.csv")
test = pd.read_csv(DATA_DIR / "application_test.csv")

print("Train:", train.shape)
print("Test :", test.shape)

Train: (307511, 122)
Test : (48744, 121)


In [10]:
train_model = train.merge(
    previous_features,
    on="SK_ID_CURR",
    how="left"
)

test_model = test.merge(
    previous_features,
    on="SK_ID_CURR",
    how="left"
)

print("Train:", train_model.shape)
print("Test :", test_model.shape)

print(
    "Applicants with previous history:",
    train_model["prev_app_count"].notna().sum()
)

print(
    "Applicants without previous history:",
    train_model["prev_app_count"].isna().sum()
)

Train: (307511, 138)
Test : (48744, 137)
Applicants with previous history: 291057
Applicants without previous history: 16454


In [11]:
%pip install lightgbm

Note: you may need to restart the kernel to use updated packages.


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

TARGET = "TARGET"
ID_COL = "SK_ID_CURR"

X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True
                ))
            ]),
            categorical_features
        )
    ]
)

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

logistic_model.fit(X_train, y_train)

valid_prob = logistic_model.predict_proba(X_valid)[:, 1]

e1_auc = roc_auc_score(y_valid, valid_prob)
e1_pr_auc = average_precision_score(y_valid, valid_prob)

print("E1 ROC-AUC :", round(e1_auc, 4))
print("E1 PR-AUC  :", round(e1_pr_auc, 4))

/opt/anaconda3/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


E1 ROC-AUC : 0.7487
E1 PR-AUC  : 0.228


In [13]:

# E2 — APPLICATION + PREVIOUS APPLICATIONS


X = train_model.drop(columns=["TARGET", "SK_ID_CURR"])
y = train_model["TARGET"]

X_train_2, X_valid_2, y_train_2, y_valid_2 = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

numeric_features_2 = X_train_2.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features_2 = X_train_2.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numeric features:", len(numeric_features_2))
print("Categorical features:", len(categorical_features_2))
print("Total features:", len(X.columns))

Numeric features: 120
Categorical features: 16
Total features: 136


In [14]:
preprocessor_2 = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features_2
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True
                ))
            ]),
            categorical_features_2
        )
    ]
)

logistic_model_2 = Pipeline([
    ("preprocessor", preprocessor_2),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

logistic_model_2.fit(X_train_2, y_train_2)

valid_prob_2 = logistic_model_2.predict_proba(
    X_valid_2
)[:, 1]

e2_auc = roc_auc_score(
    y_valid_2,
    valid_prob_2
)

e2_pr_auc = average_precision_score(
    y_valid_2,
    valid_prob_2
)

print("E2 ROC-AUC :", round(e2_auc, 4))
print("E2 PR-AUC  :", round(e2_pr_auc, 4))

E2 ROC-AUC : 0.7584
E2 PR-AUC  : 0.2388


In [15]:
print("ROC-AUC improvement:",
      round(e2_auc - e1_auc, 4))

print("PR-AUC improvement:",
      round(e2_pr_auc - e1_pr_auc, 4))

ROC-AUC improvement: 0.0097
PR-AUC improvement: 0.0108


In [16]:
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# Use the SAME E2 split for fair comparison
lgbm_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

# LightGBM needs categorical columns encoded as category
X_train_lgb = X_train_2.copy()
X_valid_lgb = X_valid_2.copy()

categorical_features_lgb = X_train_lgb.select_dtypes(
    exclude=np.number
).columns.tolist()

for col in categorical_features_lgb:
    X_train_lgb[col] = X_train_lgb[col].astype("category")

    # Make validation categories match training categories
    X_valid_lgb[col] = X_valid_lgb[col].astype(
        pd.CategoricalDtype(
            categories=X_train_lgb[col].cat.categories
        )
    )

lgbm_model.fit(
    X_train_lgb,
    y_train_2,
    categorical_feature=categorical_features_lgb
)

valid_prob_3 = lgbm_model.predict_proba(X_valid_lgb)[:, 1]

e3_auc = roc_auc_score(y_valid_2, valid_prob_3)
e3_pr_auc = average_precision_score(y_valid_2, valid_prob_3)

print("E3 ROC-AUC :", round(e3_auc, 4))
print("E3 PR-AUC  :", round(e3_pr_auc, 4))

print()
print("vs E2 ROC-AUC improvement:",
      round(e3_auc - e2_auc, 4))

print("vs E2 PR-AUC improvement:",
      round(e3_pr_auc - e2_pr_auc, 4))

[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.034570 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13896
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 132
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432482
[LightGBM] [Info] Start training from score -2.432482
E3 ROC-AUC : 0.7708
E3 PR-AUC  : 0.2611

vs E2 ROC-AUC improvement: 0.0124
vs E2 PR-AUC improvement: 0.0223


In [17]:
bureau_query = f"""
SELECT
    SK_ID_CURR,

    COUNT(*) AS bureau_credit_count,

    SUM(CASE
        WHEN CREDIT_ACTIVE = 'Active' THEN 1
        ELSE 0
    END) AS bureau_active_credit_count,

    SUM(CASE
        WHEN CREDIT_ACTIVE = 'Closed' THEN 1
        ELSE 0
    END) AS bureau_closed_credit_count,

    SUM(CASE
        WHEN CREDIT_ACTIVE = 'Bad debt' THEN 1
        ELSE 0
    END) AS bureau_bad_debt_count,

    SUM(CASE
        WHEN CREDIT_ACTIVE = 'Sold' THEN 1
        ELSE 0
    END) AS bureau_sold_credit_count,

    AVG(AMT_CREDIT_SUM) AS bureau_avg_credit,

    SUM(AMT_CREDIT_SUM) AS bureau_total_credit,

    MAX(AMT_CREDIT_SUM) AS bureau_max_credit,

    AVG(AMT_CREDIT_SUM_DEBT) AS bureau_avg_debt,

    SUM(AMT_CREDIT_SUM_DEBT) AS bureau_total_debt,

    MAX(AMT_CREDIT_SUM_DEBT) AS bureau_max_debt,

    AVG(AMT_CREDIT_SUM_OVERDUE) AS bureau_avg_overdue,

    SUM(AMT_CREDIT_SUM_OVERDUE) AS bureau_total_overdue,

    MAX(AMT_CREDIT_SUM_OVERDUE) AS bureau_max_overdue,

    AVG(
        CASE
            WHEN AMT_CREDIT_SUM > 0
            THEN AMT_CREDIT_SUM_DEBT / AMT_CREDIT_SUM
            ELSE NULL
        END
    ) AS bureau_avg_debt_to_credit,

    AVG(DAYS_CREDIT) AS bureau_avg_days_credit,

    MIN(DAYS_CREDIT) AS bureau_oldest_credit_days,

    MAX(DAYS_CREDIT) AS bureau_most_recent_credit_days,

    AVG(CREDIT_DAY_OVERDUE) AS bureau_avg_days_overdue,

    MAX(CREDIT_DAY_OVERDUE) AS bureau_max_days_overdue,

    AVG(CNT_CREDIT_PROLONG) AS bureau_avg_credit_prolongations,

    SUM(CASE
        WHEN CREDIT_DAY_OVERDUE > 0 THEN 1
        ELSE 0
    END) AS bureau_overdue_account_count

FROM read_csv_auto('../data/raw/bureau.csv')

GROUP BY SK_ID_CURR
"""

bureau_features = con.sql(bureau_query).df()

print("Bureau features shape:", bureau_features.shape)
bureau_features.head()

Bureau features shape: (305811, 23)


,SK_ID_CURR,bureau_credit_count,bureau_active_credit_count,bureau_closed_credit_count,bureau_bad_debt_count,bureau_sold_credit_count,bureau_avg_credit,bureau_total_credit,bureau_max_credit,bureau_avg_debt,...,bureau_total_overdue,bureau_max_overdue,bureau_avg_debt_to_credit,bureau_avg_days_credit,bureau_oldest_credit_days,bureau_most_recent_credit_days,bureau_avg_days_overdue,bureau_max_days_overdue,bureau_avg_credit_prolongations,bureau_overdue_account_count
0,321485,6,5.0,1.0,0.0,0.0,96935.617500,581613.705,292500.0,74334.750000,...,0.0,0.0,0.643130,-587.166667,-2306,-112,0.0,0,0.0,0.0
1,135101,9,3.0,6.0,0.0,0.0,167360.075000,1506240.675,553500.0,41540.000000,...,0.0,0.0,0.124641,-1868.333333,-2814,-344,0.0,0,0.0,0.0
2,214301,7,3.0,4.0,0.0,0.0,241275.111429,1688925.780,675000.0,89868.214286,...,0.0,0.0,0.258246,-1485.714286,-2673,-179,0.0,0,0.0,0.0
3,394701,5,3.0,2.0,0.0,0.0,675904.572000,3379522.860,2160000.0,524470.923000,...,0.0,0.0,0.533507,-460.600000,-785,-242,0.0,0,0.0,0.0
4,410383,7,0.0,7.0,0.0,0.0,101154.535714,708081.750,161190.0,0.000000,...,0.0,0.0,0.000000,-1963.142857,-2757,-480,0.0,0,0.0,0.0


In [18]:
bureau_features.to_parquet(
    processed_dir / "bureau_features.parquet",
    index=False
)

print("Saved bureau features.")

Saved bureau features.


In [19]:
train_model_e4 = train_model.merge(
    bureau_features,
    on="SK_ID_CURR",
    how="left"
)

test_model_e4 = test_model.merge(
    bureau_features,
    on="SK_ID_CURR",
    how="left"
)

print("E4 train shape:", train_model_e4.shape)
print("E4 test shape :", test_model_e4.shape)

print(
    "Applicants with bureau history:",
    train_model_e4["bureau_credit_count"].notna().sum()
)

print(
    "Applicants without bureau history:",
    train_model_e4["bureau_credit_count"].isna().sum()
)

E4 train shape: (307511, 160)
E4 test shape : (48744, 159)
Applicants with bureau history: 263491
Applicants without bureau history: 44020


In [20]:
# Additional bureau risk ratios
train_model_e4["bureau_debt_to_credit_total"] = (
    train_model_e4["bureau_total_debt"]
    / train_model_e4["bureau_total_credit"].replace(0, np.nan)
)

test_model_e4["bureau_debt_to_credit_total"] = (
    test_model_e4["bureau_total_debt"]
    / test_model_e4["bureau_total_credit"].replace(0, np.nan)
)

train_model_e4["bureau_overdue_to_credit"] = (
    train_model_e4["bureau_total_overdue"]
    / train_model_e4["bureau_total_credit"].replace(0, np.nan)
)

test_model_e4["bureau_overdue_to_credit"] = (
    test_model_e4["bureau_total_overdue"]
    / test_model_e4["bureau_total_credit"].replace(0, np.nan)
)

train_model_e4["bureau_active_ratio"] = (
    train_model_e4["bureau_active_credit_count"]
    / train_model_e4["bureau_credit_count"].replace(0, np.nan)
)

test_model_e4["bureau_active_ratio"] = (
    test_model_e4["bureau_active_credit_count"]
    / test_model_e4["bureau_credit_count"].replace(0, np.nan)
)

train_model_e4["bureau_overdue_account_ratio"] = (
    train_model_e4["bureau_overdue_account_count"]
    / train_model_e4["bureau_credit_count"].replace(0, np.nan)
)

test_model_e4["bureau_overdue_account_ratio"] = (
    test_model_e4["bureau_overdue_account_count"]
    / test_model_e4["bureau_credit_count"].replace(0, np.nan)
)

print("Updated E4 shape:", train_model_e4.shape)

Updated E4 shape: (307511, 164)


/var/folders/jj/1fp60rtn35g1z319tfv84jcr0000gn/T/ipykernel_9625/1363585914.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_model_e4["bureau_debt_to_credit_total"] = (
/var/folders/jj/1fp60rtn35g1z319tfv84jcr0000gn/T/ipykernel_9625/1363585914.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_model_e4["bureau_debt_to_credit_total"] = (
/var/folders/jj/1fp60rtn35g1z319tfv84jcr0000gn/T/ipykernel_9625/1363585914.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.i

In [21]:
X_e4 = train_model_e4.drop(
    columns=["TARGET", "SK_ID_CURR"]
)
y_e4 = train_model_e4["TARGET"]

X_train_e4, X_valid_e4, y_train_e4, y_valid_e4 = train_test_split(
    X_e4,
    y_e4,
    test_size=0.20,
    stratify=y_e4,
    random_state=42
)

categorical_features_e4 = X_train_e4.select_dtypes(
    exclude=np.number
).columns.tolist()

for col in categorical_features_e4:
    X_train_e4[col] = X_train_e4[col].astype("category")

    X_valid_e4[col] = X_valid_e4[col].astype(
        pd.CategoricalDtype(
            categories=X_train_e4[col].cat.categories
        )
    )

print("Training shape:", X_train_e4.shape)
print("Validation shape:", X_valid_e4.shape)
print("Categorical features:", len(categorical_features_e4))

Training shape: (246008, 162)
Validation shape: (61503, 162)
Categorical features: 16


In [22]:
lgbm_e4 = LGBMClassifier(
    n_estimators=600,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

lgbm_e4.fit(
    X_train_e4,
    y_train_e4,
    categorical_feature=categorical_features_e4
)

valid_prob_e4 = lgbm_e4.predict_proba(
    X_valid_e4
)[:, 1]

e4_auc = roc_auc_score(
    y_valid_e4,
    valid_prob_e4
)

e4_pr_auc = average_precision_score(
    y_valid_e4,
    valid_prob_e4
)

print("E4 ROC-AUC :", round(e4_auc, 4))
print("E4 PR-AUC  :", round(e4_pr_auc, 4))

print()
print("vs E3 ROC-AUC improvement:",
      round(e4_auc - e3_auc, 4))

print("vs E3 PR-AUC improvement:",
      round(e4_pr_auc - e3_pr_auc, 4))

[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.032519 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 18603
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 158
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432482
[LightGBM] [Info] Start training from score -2.432482
E4 ROC-AUC : 0.7746
E4 PR-AUC  : 0.2687

vs E3 ROC-AUC improvement: 0.0039
vs E3 PR-AUC improvement: 0.0076


In [23]:
installments_query = """
SELECT
    SK_ID_CURR,

    -- Volume
    COUNT(*) AS installment_count,

    COUNT(DISTINCT SK_ID_PREV) AS installment_loan_count,

    -- Payment timing
    AVG(DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT)
        AS avg_payment_delay,

    MAX(DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT)
        AS max_payment_delay,

    MIN(DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT)
        AS min_payment_delay,

    -- Late payments
    SUM(
        CASE
            WHEN DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT
            THEN 1 ELSE 0
        END
    ) AS late_payment_count,

    AVG(
        CASE
            WHEN DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT
            THEN 1.0 ELSE 0.0
        END
    ) AS late_payment_rate,

    -- Payment amount behavior
    AVG(AMT_INSTALMENT) AS avg_installment_amount,

    AVG(AMT_PAYMENT) AS avg_payment_amount,

    MAX(AMT_PAYMENT) AS max_payment_amount,

    -- Payment completion ratio
    AVG(
        CASE
            WHEN AMT_INSTALMENT > 0
            THEN AMT_PAYMENT / AMT_INSTALMENT
            ELSE NULL
        END
    ) AS avg_payment_completion_ratio,

    -- Underpayment
    SUM(
        CASE
            WHEN AMT_PAYMENT < AMT_INSTALMENT
            THEN 1 ELSE 0
        END
    ) AS underpayment_count,

    AVG(
        CASE
            WHEN AMT_PAYMENT < AMT_INSTALMENT
            THEN 1.0 ELSE 0.0
        END
    ) AS underpayment_rate,

    -- Payment surplus
    AVG(
        CASE
            WHEN AMT_INSTALMENT > 0
            THEN (AMT_PAYMENT - AMT_INSTALMENT) / AMT_INSTALMENT
            ELSE NULL
        END
    ) AS avg_payment_surplus_ratio

FROM read_csv_auto(
    '../data/raw/installments_payments.csv'
)

GROUP BY SK_ID_CURR
"""

installment_features = con.sql(installments_query).df()

print("Installment features shape:", installment_features.shape)
installment_features.head()

Installment features shape: (339587, 15)


,SK_ID_CURR,installment_count,installment_loan_count,avg_payment_delay,max_payment_delay,min_payment_delay,late_payment_count,late_payment_rate,avg_installment_amount,avg_payment_amount,max_payment_amount,avg_payment_completion_ratio,underpayment_count,underpayment_rate,avg_payment_surplus_ratio
0,193008,103,4,-6.446602,1.0,-23.0,1.0,0.009709,6127.587961,6127.587961,36000.000,1.0,0.0,0.0,0.0
1,100200,23,3,-18.217391,-4.0,-38.0,0.0,0.000000,8269.808478,8269.808478,14649.750,1.0,0.0,0.0,0.0
2,113852,9,1,-15.444444,-9.0,-23.0,0.0,0.000000,7632.135000,7632.135000,7632.135,1.0,0.0,0.0,0.0
3,131941,69,3,-5.159420,0.0,-22.0,0.0,0.000000,6871.275652,6871.275652,67500.000,1.0,0.0,0.0,0.0
4,133624,47,2,-0.829787,0.0,-15.0,0.0,0.000000,3450.580851,3450.580851,23652.180,1.0,0.0,0.0,0.0


In [24]:
installment_features.to_parquet(
    processed_dir / "installment_features.parquet",
    index=False
)

print("Saved installment features.")

Saved installment features.


In [25]:
train_model_e5 = train_model_e4.merge(
    installment_features,
    on="SK_ID_CURR",
    how="left"
)

test_model_e5 = test_model_e4.merge(
    installment_features,
    on="SK_ID_CURR",
    how="left"
)

print("E5 train shape:", train_model_e5.shape)
print("E5 test shape :", test_model_e5.shape)

print(
    "Applicants with installment history:",
    train_model_e5["installment_count"].notna().sum()
)

print(
    "Applicants without installment history:",
    train_model_e5["installment_count"].isna().sum()
)

E5 train shape: (307511, 178)
E5 test shape : (48744, 177)
Applicants with installment history: 291643
Applicants without installment history: 15868


In [26]:
X_e5 = train_model_e5.drop(
    columns=["TARGET", "SK_ID_CURR"]
)

y_e5 = train_model_e5["TARGET"]

X_train_e5, X_valid_e5, y_train_e5, y_valid_e5 = train_test_split(
    X_e5,
    y_e5,
    test_size=0.20,
    stratify=y_e5,
    random_state=42
)

categorical_features_e5 = X_train_e5.select_dtypes(
    exclude=np.number
).columns.tolist()

for col in categorical_features_e5:
    X_train_e5[col] = X_train_e5[col].astype("category")

    X_valid_e5[col] = X_valid_e5[col].astype(
        pd.CategoricalDtype(
            categories=X_train_e5[col].cat.categories
        )
    )

print("Training shape:", X_train_e5.shape)
print("Validation shape:", X_valid_e5.shape)
print("Categorical features:", len(categorical_features_e5))

Training shape: (246008, 176)
Validation shape: (61503, 176)
Categorical features: 16


In [27]:
lgbm_e5 = LGBMClassifier(
    n_estimators=600,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

lgbm_e5.fit(
    X_train_e5,
    y_train_e5,
    categorical_feature=categorical_features_e5
)

valid_prob_e5 = lgbm_e5.predict_proba(
    X_valid_e5
)[:, 1]

e5_auc = roc_auc_score(
    y_valid_e5,
    valid_prob_e5
)

e5_pr_auc = average_precision_score(
    y_valid_e5,
    valid_prob_e5
)

print("E5 ROC-AUC :", round(e5_auc, 4))
print("E5 PR-AUC  :", round(e5_pr_auc, 4))

print()
print("vs E4 ROC-AUC improvement:",
      round(e5_auc - e4_auc, 4))

print("vs E4 PR-AUC improvement:",
      round(e5_pr_auc - e4_pr_auc, 4))

[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.034107 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 21520
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 172
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432482
[LightGBM] [Info] Start training from score -2.432482
E5 ROC-AUC : 0.7796
E5 PR-AUC  : 0.2773

vs E4 ROC-AUC improvement: 0.0049
vs E4 PR-AUC improvement: 0.0085


In [28]:
pos_query = """
SELECT
    SK_ID_CURR,

    COUNT(*) AS pos_record_count,

    COUNT(DISTINCT SK_ID_PREV) AS pos_loan_count,

    COUNT(DISTINCT NAME_CONTRACT_STATUS) AS pos_status_count,

    SUM(
        CASE
            WHEN NAME_CONTRACT_STATUS = 'Active'
            THEN 1 ELSE 0
        END
    ) AS pos_active_count,

    SUM(
        CASE
            WHEN NAME_CONTRACT_STATUS = 'Completed'
            THEN 1 ELSE 0
        END
    ) AS pos_completed_count,

    SUM(
        CASE
            WHEN SK_DPD > 0
            THEN 1 ELSE 0
        END
    ) AS pos_dpd_record_count,

    AVG(SK_DPD) AS pos_avg_dpd,

    MAX(SK_DPD) AS pos_max_dpd,

    AVG(SK_DPD_DEF) AS pos_avg_dpd_def,

    MAX(SK_DPD_DEF) AS pos_max_dpd_def,

    AVG(CNT_INSTALMENT) AS pos_avg_installments,

    AVG(CNT_INSTALMENT_FUTURE)
        AS pos_avg_future_installments,

    MAX(CNT_INSTALMENT_FUTURE)
        AS pos_max_future_installments

FROM read_csv_auto(
    '../data/raw/POS_CASH_balance.csv'
)

GROUP BY SK_ID_CURR
"""

pos_features = con.sql(pos_query).df()

print("POS features shape:", pos_features.shape)
pos_features.head()

POS features shape: (337252, 14)


,SK_ID_CURR,pos_record_count,pos_loan_count,pos_status_count,pos_active_count,pos_completed_count,pos_dpd_record_count,pos_avg_dpd,pos_max_dpd,pos_avg_dpd_def,pos_max_dpd_def,pos_avg_installments,pos_avg_future_installments,pos_max_future_installments
0,141289,16,2,2,15.0,1.0,0.0,0.0,0,0.0,0,36.375000,32.812500,60.0
1,358625,17,2,2,15.0,2.0,0.0,0.0,0,0.0,0,11.058824,6.882353,12.0
2,296532,32,1,1,32.0,0.0,0.0,0.0,0,0.0,0,42.000000,26.500000,42.0
3,306984,18,3,2,15.0,3.0,0.0,0.0,0,0.0,0,39.166667,36.444444,48.0
4,249586,23,3,3,20.0,2.0,0.0,0.0,0,0.0,0,10.826087,6.391304,12.0


In [29]:
pos_features.to_parquet(
    processed_dir / "pos_cash_features.parquet",
    index=False
)

print("Saved POS/Cash features.")

Saved POS/Cash features.


In [30]:
credit_card_query = """
SELECT
    SK_ID_CURR,

    COUNT(*) AS cc_record_count,

    COUNT(DISTINCT SK_ID_PREV) AS cc_card_count,

    AVG(AMT_BALANCE) AS cc_avg_balance,

    MAX(AMT_BALANCE) AS cc_max_balance,

    AVG(AMT_CREDIT_LIMIT_ACTUAL)
        AS cc_avg_credit_limit,

    MAX(AMT_CREDIT_LIMIT_ACTUAL)
        AS cc_max_credit_limit,

    AVG(
        CASE
            WHEN AMT_CREDIT_LIMIT_ACTUAL > 0
            THEN AMT_BALANCE / AMT_CREDIT_LIMIT_ACTUAL
            ELSE NULL
        END
    ) AS cc_avg_utilization,

    MAX(
        CASE
            WHEN AMT_CREDIT_LIMIT_ACTUAL > 0
            THEN AMT_BALANCE / AMT_CREDIT_LIMIT_ACTUAL
            ELSE NULL
        END
    ) AS cc_max_utilization,

    AVG(AMT_DRAWINGS_CURRENT)
        AS cc_avg_drawings,

    MAX(AMT_DRAWINGS_CURRENT)
        AS cc_max_drawings,

    AVG(AMT_PAYMENT_CURRENT)
        AS cc_avg_payment,

    MAX(AMT_PAYMENT_CURRENT)
        AS cc_max_payment,

    AVG(AMT_TOTAL_RECEIVABLE)
        AS cc_avg_receivable,

    MAX(AMT_TOTAL_RECEIVABLE)
        AS cc_max_receivable,

    AVG(SK_DPD) AS cc_avg_dpd,

    MAX(SK_DPD) AS cc_max_dpd,

    SUM(
        CASE
            WHEN SK_DPD > 0
            THEN 1 ELSE 0
        END
    ) AS cc_dpd_record_count,

    AVG(
        CASE
            WHEN AMT_PAYMENT_CURRENT > 0
            THEN AMT_PAYMENT_CURRENT /
                 NULLIF(AMT_TOTAL_RECEIVABLE, 0)
            ELSE NULL
        END
    ) AS cc_payment_to_receivable_ratio

FROM read_csv_auto(
    '../data/raw/credit_card_balance.csv'
)

GROUP BY SK_ID_CURR
"""

credit_card_features = con.sql(
    credit_card_query
).df()

print(
    "Credit-card features shape:",
    credit_card_features.shape
)

credit_card_features.head()

Credit-card features shape: (103558, 19)


,SK_ID_CURR,cc_record_count,cc_card_count,cc_avg_balance,cc_max_balance,cc_avg_credit_limit,cc_max_credit_limit,cc_avg_utilization,cc_max_utilization,cc_avg_drawings,cc_max_drawings,cc_avg_payment,cc_max_payment,cc_avg_receivable,cc_max_receivable,cc_avg_dpd,cc_max_dpd,cc_dpd_record_count,cc_payment_to_receivable_ratio
0,425315,91,1,28442.287582,226703.340,116208.791209,225000,0.170323,1.007570,12616.519945,229860.0,14380.337967,135000.0,28352.534835,226703.340,0.010989,1,1.0,1.315220
1,401149,96,1,27256.736719,119774.295,99140.625000,112500,0.242282,1.064660,1640.625000,112500.0,2930.439844,18000.0,27243.337500,119774.295,0.010417,1,1.0,-0.051379
2,333256,96,1,3624.182344,116553.510,35156.250000,112500,0.103088,1.036031,33.234375,2250.0,1507.536563,116595.0,3606.682031,116553.510,0.000000,0,0.0,-0.711344
3,313277,96,1,4199.615625,70115.175,62812.500000,67500,0.062236,1.038743,21.093750,2025.0,1080.087188,22500.0,4200.278438,70115.175,0.020833,1,2.0,12.207253
4,191933,96,1,14199.025313,102516.705,36562.500000,135000,0.388349,0.759383,0.000000,0.0,1879.335937,13500.0,14198.890313,102516.705,0.000000,0,0.0,3.245008


In [31]:
credit_card_features.to_parquet(
    processed_dir / "credit_card_features.parquet",
    index=False
)

print("Saved credit-card features.")

Saved credit-card features.


In [32]:
train_model_e6 = (
    train_model_e5
    .merge(pos_features, on="SK_ID_CURR", how="left")
    .merge(credit_card_features, on="SK_ID_CURR", how="left")
)

test_model_e6 = (
    test_model_e5
    .merge(pos_features, on="SK_ID_CURR", how="left")
    .merge(credit_card_features, on="SK_ID_CURR", how="left")
)

print("E6 train shape:", train_model_e6.shape)
print("E6 test shape :", test_model_e6.shape)

print(
    "Applicants with POS history:",
    train_model_e6["pos_record_count"].notna().sum()
)

print(
    "Applicants with credit-card history:",
    train_model_e6["cc_record_count"].notna().sum()
)

E6 train shape: (307511, 209)
E6 test shape : (48744, 208)
Applicants with POS history: 289444
Applicants with credit-card history: 86905


In [33]:
X_e6 = train_model_e6.drop(
    columns=["TARGET", "SK_ID_CURR"]
)

y_e6 = train_model_e6["TARGET"]

X_train_e6, X_valid_e6, y_train_e6, y_valid_e6 = train_test_split(
    X_e6,
    y_e6,
    test_size=0.20,
    stratify=y_e6,
    random_state=42
)

categorical_features_e6 = X_train_e6.select_dtypes(
    exclude=np.number
).columns.tolist()

for col in categorical_features_e6:
    X_train_e6[col] = X_train_e6[col].astype("category")

    X_valid_e6[col] = X_valid_e6[col].astype(
        pd.CategoricalDtype(
            categories=X_train_e6[col].cat.categories
        )
    )

print("Training shape:", X_train_e6.shape)
print("Validation shape:", X_valid_e6.shape)
print("Categorical features:", len(categorical_features_e6))

Training shape: (246008, 207)
Validation shape: (61503, 207)
Categorical features: 16


In [34]:
lgbm_e6 = LGBMClassifier(
    n_estimators=600,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

lgbm_e6.fit(
    X_train_e6,
    y_train_e6,
    categorical_feature=categorical_features_e6
)

valid_prob_e6 = lgbm_e6.predict_proba(
    X_valid_e6
)[:, 1]

e6_auc = roc_auc_score(
    y_valid_e6,
    valid_prob_e6
)

e6_pr_auc = average_precision_score(
    y_valid_e6,
    valid_prob_e6
)

print("E6 ROC-AUC :", round(e6_auc, 4))
print("E6 PR-AUC  :", round(e6_pr_auc, 4))

print()
print("vs E5 ROC-AUC improvement:",
      round(e6_auc - e5_auc, 4))

print("vs E5 PR-AUC improvement:",
      round(e6_pr_auc - e5_pr_auc, 4))

[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.040707 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 27298
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 203
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432482
[LightGBM] [Info] Start training from score -2.432482
E6 ROC-AUC : 0.7837
E6 PR-AUC  : 0.2862

vs E5 ROC-AUC improvement: 0.0041
vs E5 PR-AUC improvement: 0.0089


In [35]:
%pip install xgboost


Note: you may need to restart the kernel to use updated packages.


In [36]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

X_train_xgb = X_train_e6.copy()
X_valid_xgb = X_valid_e6.copy()

for col in categorical_features_e6:

    # Use the exact training categories
    categories = X_train_xgb[col].cat.categories

    # Convert both to strings temporarily
    train_values = X_train_xgb[col].astype("object")
    valid_values = X_valid_xgb[col].astype("object")

    # Create category -> integer mapping
    category_map = {
        category: i
        for i, category in enumerate(categories)
    }

    # Convert to numeric codes
    X_train_xgb[col] = (
        train_values
        .map(category_map)
        .fillna(-1)
        .astype("int32")
    )

    X_valid_xgb[col] = (
        valid_values
        .map(category_map)
        .fillna(-1)
        .astype("int32")
    )

print("Training shape:", X_train_xgb.shape)
print("Validation shape:", X_valid_xgb.shape)
print("Categorical columns converted:", len(categorical_features_e6))

Training shape: (246008, 207)
Validation shape: (61503, 207)
Categorical columns converted: 16


In [37]:
scale_pos_weight = (
    (y_train_e6 == 0).sum()
    / (y_train_e6 == 1).sum()
)

print("scale_pos_weight:", round(scale_pos_weight, 2))

scale_pos_weight: 11.39


In [38]:
xgb_e6 = XGBClassifier(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=2.0,
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_e6.fit(

    X_train_xgb,
    y_train_e6,
    eval_set=[(X_valid_xgb, y_valid_e6)],
    verbose=False
)

KeyboardInterrupt: 

In [ ]:
valid_prob_xgb = xgb_e6.predict_proba(
    X_valid_xgb
)[:, 1]

xgb_auc = roc_auc_score(
    y_valid_e6,
    valid_prob_xgb
)

xgb_pr_auc = average_precision_score(
    y_valid_e6,
    valid_prob_xgb
)

print("XGBoost ROC-AUC :", round(xgb_auc, 4))
print("XGBoost PR-AUC  :", round(xgb_pr_auc, 4))

print()
print("vs E6 LightGBM ROC-AUC:",
      round(xgb_auc - e6_auc, 4))

print("vs E6 LightGBM PR-AUC:",
      round(xgb_pr_auc - e6_pr_auc, 4))

In [ ]:
import numpy as np
import pandas as pd

train_fe = train_model_e6.copy()
test_fe = test_model_e6.copy()


def add_application_features(df):

    # -------------------------
    # Age / employment
    # -------------------------

    df["AGE_YEARS"] = (-df["DAYS_BIRTH"] / 365.25)

    df["EMPLOYED_YEARS"] = np.where(
        df["DAYS_EMPLOYED"] < 0,
        -df["DAYS_EMPLOYED"] / 365.25,
        np.nan
    )

    df["EMPLOYMENT_AGE_RATIO"] = (
        df["EMPLOYED_YEARS"] /
        df["AGE_YEARS"].replace(0, np.nan)
    )

    # -------------------------
    # Credit / income
    # -------------------------

    df["CREDIT_INCOME_RATIO"] = (
        df["AMT_CREDIT"] /
        df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    )

    df["ANNUITY_INCOME_RATIO"] = (
        df["AMT_ANNUITY"] /
        df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    )

    df["CREDIT_ANNUITY_RATIO"] = (
        df["AMT_CREDIT"] /
        df["AMT_ANNUITY"].replace(0, np.nan)
    )

    df["GOODS_CREDIT_RATIO"] = (
        df["AMT_GOODS_PRICE"] /
        df["AMT_CREDIT"].replace(0, np.nan)
    )

    df["INCOME_CREDIT_RATIO"] = (
        df["AMT_INCOME_TOTAL"] /
        df["AMT_CREDIT"].replace(0, np.nan)
    )

    # -------------------------
    # Family adjusted measures
    # -------------------------

    family_size = df["CNT_FAM_MEMBERS"].replace(0, np.nan)

    df["INCOME_PER_FAMILY_MEMBER"] = (
        df["AMT_INCOME_TOTAL"] / family_size
    )

    df["CREDIT_PER_FAMILY_MEMBER"] = (
        df["AMT_CREDIT"] / family_size
    )

    df["ANNUITY_PER_FAMILY_MEMBER"] = (
        df["AMT_ANNUITY"] / family_size
    )

    df["INCOME_PER_CHILD"] = (
        df["AMT_INCOME_TOTAL"] /
        (df["CNT_CHILDREN"] + 1)
    )

    # -------------------------
    # External risk scores
    # -------------------------

    ext_cols = [
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3"
    ]

    df["EXT_SOURCE_MEAN"] = df[ext_cols].mean(axis=1)
    df["EXT_SOURCE_MIN"] = df[ext_cols].min(axis=1)
    df["EXT_SOURCE_MAX"] = df[ext_cols].max(axis=1)
    df["EXT_SOURCE_STD"] = df[ext_cols].std(axis=1)

    df["EXT_SOURCE_PRODUCT"] = (
        df["EXT_SOURCE_1"] *
        df["EXT_SOURCE_2"] *
        df["EXT_SOURCE_3"]
    )

    df["EXT_SOURCE_WEIGHTED"] = (
        0.5 * df["EXT_SOURCE_2"] +
        0.3 * df["EXT_SOURCE_3"] +
        0.2 * df["EXT_SOURCE_1"]
    )

    # -------------------------
    # Credit stress interactions
    # -------------------------

    df["CREDIT_STRESS_EXT"] = (
        df["CREDIT_INCOME_RATIO"] *
        (1 - df["EXT_SOURCE_MEAN"])
    )

    df["ANNUITY_STRESS_EXT"] = (
        df["ANNUITY_INCOME_RATIO"] *
        (1 - df["EXT_SOURCE_MEAN"])
    )

    return df


train_fe = add_application_features(train_fe)
test_fe = add_application_features(test_fe)

print("Original E6 columns:", train_model_e6.shape[1])
print("New columns:", train_fe.shape[1])
print("Added:", train_fe.shape[1] - train_model_e6.shape[1])

In [ ]:
X_fe = train_fe.drop(
    columns=["TARGET", "SK_ID_CURR"]
)

y_fe = train_fe["TARGET"]

X_train_fe, X_valid_fe, y_train_fe, y_valid_fe = train_test_split(
    X_fe,
    y_fe,
    test_size=0.20,
    stratify=y_fe,
    random_state=42
)

categorical_features_fe = X_train_fe.select_dtypes(
    exclude=np.number
).columns.tolist()

for col in categorical_features_fe:

    X_train_fe[col] = X_train_fe[col].astype("category")

    X_valid_fe[col] = X_valid_fe[col].astype(
        pd.CategoricalDtype(
            categories=X_train_fe[col].cat.categories
        )
    )

print(X_train_fe.shape)
print(X_valid_fe.shape)

In [ ]:
lgbm_fe = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.025,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=30,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=-1
)

lgbm_fe.fit(
    X_train_fe,
    y_train_fe,
    categorical_feature=categorical_features_fe
)

valid_prob_fe = lgbm_fe.predict_proba(
    X_valid_fe
)[:, 1]

fe_auc = roc_auc_score(
    y_valid_fe,
    valid_prob_fe
)

fe_pr_auc = average_precision_score(
    y_valid_fe,
    valid_prob_fe
)

print("Feature-engineered LightGBM ROC-AUC:", round(fe_auc, 4))
print("Feature-engineered LightGBM PR-AUC :", round(fe_pr_auc, 4))

print()
print("vs E6:", round(fe_auc - e6_auc, 4))

In [ ]:
%pip install catboost


In [ ]:
from catboost import CatBoostClassifier

X_train_cat = X_train_fe.copy()
X_valid_cat = X_valid_fe.copy()

# Convert categorical columns to clean strings
for col in categorical_features_fe:
    X_train_cat[col] = (
        X_train_cat[col]
        .astype("object")
        .where(X_train_cat[col].notna(), "MISSING")
        .astype(str)
    )

    X_valid_cat[col] = (
        X_valid_cat[col]
        .astype("object")
        .where(X_valid_cat[col].notna(), "MISSING")
        .astype(str)
    )

cat_features_idx = [
    X_train_cat.columns.get_loc(col)
    for col in categorical_features_fe
]

print("Training:", X_train_cat.shape)
print("Validation:", X_valid_cat.shape)
print("Categorical features:", len(cat_features_idx))

# Verify there are NO missing values in categorical columns
print(
    "Categorical NaNs:",
    X_train_cat[categorical_features_fe].isna().sum().sum(),
    X_valid_cat[categorical_features_fe].isna().sum().sum()
)

In [ ]:
cat_model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.035,
    depth=7,
    loss_function="Logloss",
    eval_metric="AUC",
    l2_leaf_reg=5,
    random_seed=42,
    thread_count=-1,
    verbose=100,
    allow_writing_files=False
)

cat_model.fit(
    X_train_cat,
    y_train_fe,
    cat_features=cat_features_idx,
    eval_set=(X_valid_cat, y_valid_fe),
    use_best_model=True,
    early_stopping_rounds=100
)

In [ ]:
valid_prob_cat = cat_model.predict_proba(
    X_valid_cat
)[:, 1]

cat_auc = roc_auc_score(
    y_valid_fe,
    valid_prob_cat
)

cat_pr_auc = average_precision_score(
    y_valid_fe,
    valid_prob_cat
)

print("CatBoost ROC-AUC :", round(cat_auc, 4))
print("CatBoost PR-AUC  :", round(cat_pr_auc, 4))

print()
print("vs LightGBM FE ROC-AUC:",
      round(cat_auc - fe_auc, 4))

print("vs LightGBM FE PR-AUC:",
      round(cat_pr_auc - fe_pr_auc, 4))

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

# Make sure all predictions correspond to the SAME validation rows
predictions = {
    "LightGBM_FE": valid_prob_fe,
    "XGBoost": valid_prob_xgb,
    "CatBoost": valid_prob_cat
}

# Test several blend weights
blend_results = []

for lgb_w in np.arange(0.0, 1.01, 0.05):
    for xgb_w in np.arange(0.0, 1.01 - lgb_w, 0.05):

        cat_w = 1.0 - lgb_w - xgb_w

        if cat_w < 0:
            continue

        blend_prob = (
            lgb_w * valid_prob_fe
            + xgb_w * valid_prob_xgb
            + cat_w * valid_prob_cat
        )

        auc = roc_auc_score(
            y_valid_fe,
            blend_prob
        )

        pr_auc = average_precision_score(
            y_valid_fe,
            blend_prob
        )

        blend_results.append({
            "LightGBM_weight": lgb_w,
            "XGBoost_weight": xgb_w,
            "CatBoost_weight": cat_w,
            "ROC_AUC": auc,
            "PR_AUC": pr_auc
        })

blend_results_df = pd.DataFrame(blend_results)

best_blend = blend_results_df.sort_values(
    "ROC_AUC",
    ascending=False
).iloc[0]

print("BEST BLEND")
print(best_blend)

In [ ]:
print("\nTop 10 blends by ROC-AUC:")

display(
    blend_results_df
    .sort_values("ROC_AUC", ascending=False)
    .head(10)
)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

def evaluate_model(name, model, X_train, y_train, X_valid, y_valid):
    train_prob = model.predict_proba(X_train)[:, 1]
    valid_prob = model.predict_proba(X_valid)[:, 1]

    print(f"\n{name}")
    print(f"Train ROC-AUC : {roc_auc_score(y_train, train_prob):.4f}")
    print(f"Valid ROC-AUC : {roc_auc_score(y_valid, valid_prob):.4f}")
    print(f"Gap           : {roc_auc_score(y_train, train_prob) - roc_auc_score(y_valid, valid_prob):.4f}")
    print(f"Train PR-AUC  : {average_precision_score(y_train, train_prob):.4f}")
    print(f"Valid PR-AUC  : {average_precision_score(y_valid, valid_prob):.4f}")

In [ ]:
evaluate_model(
    "LightGBM FE",
    lgbm_fe,
    X_train_fe,
    y_train_fe,
    X_valid_fe,
    y_valid_fe
)

evaluate_model(
    "CatBoost",
    cat_model,
    X_train_cat,
    y_train_fe,
    X_valid_cat,
    y_valid_fe
)

In [ ]:
lgbm_tuned = LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=20,
    max_depth=-1,
    min_child_samples=80,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=5.0,
    random_state=42,
    n_jobs=-1
)

In [ ]:
cat_tuned = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.025,
    depth=6,
    l2_leaf_reg=10,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    thread_count=-1,
    verbose=200,
    allow_writing_files=False
)

In [ ]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.metrics import roc_auc_score, average_precision_score

lgbm_tuned = LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=20,
    max_depth=-1,
    min_child_samples=80,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=5.0,
    random_state=42,
    n_jobs=-1
)

lgbm_tuned.fit(
    X_train_fe,
    y_train_fe,
    eval_set=[(X_valid_fe, y_valid_fe)],
    eval_metric="auc",
    callbacks=[
        early_stopping(100),
        log_evaluation(200)
    ]
)

train_prob = lgbm_tuned.predict_proba(X_train_fe)[:, 1]
valid_prob = lgbm_tuned.predict_proba(X_valid_fe)[:, 1]

train_auc = roc_auc_score(y_train_fe, train_prob)
valid_auc = roc_auc_score(y_valid_fe, valid_prob)

print("\nTUNED LIGHTGBM")
print(f"Best iteration : {lgbm_tuned.best_iteration_}")
print(f"Train ROC-AUC  : {train_auc:.4f}")
print(f"Valid ROC-AUC  : {valid_auc:.4f}")
print(f"Gap            : {train_auc - valid_auc:.4f}")
print(f"Valid PR-AUC   : {average_precision_score(y_valid_fe, valid_prob):.4f}")

In [ ]:
from sklearn.inspection import permutation_importance
import pandas as pd
import numpy as np

# CatBoost feature importance
feature_importance = pd.DataFrame({
    "feature": X_train_cat.columns,
    "importance": cat_model.get_feature_importance()
})

feature_importance = feature_importance.sort_values(
    "importance", ascending=False
).reset_index(drop=True)

print("Top 30 features:")
display(feature_importance.head(30))

In [ ]:
import matplotlib.pyplot as plt

top20 = feature_importance.head(20).sort_values("importance")

plt.figure(figsize=(10, 7))
plt.barh(top20["feature"], top20["importance"])
plt.xlabel("CatBoost Feature Importance")
plt.ylabel("Feature")
plt.title("Top 20 Features — Credit Risk Model")
plt.tight_layout()
plt.show()

In [ ]:
%pip install shap

In [ ]:
import shap
import matplotlib.pyplot as plt

# Use a manageable validation sample
X_shap = X_valid_cat.sample(
    n=min(3000, len(X_valid_cat)),
    random_state=42
)

print("SHAP sample:", X_shap.shape)

In [ ]:
explainer = shap.TreeExplainer(cat_model)

shap_values = explainer.shap_values(X_shap)

# CatBoost binary classification can return a single array
if isinstance(shap_values, list):
    shap_values = shap_values[1]

print("SHAP shape:", shap_values.shape)

In [ ]:
shap.summary_plot(
    shap_values,
    X_shap,
    max_display=20
)

In [ ]:
# Pick one validation applicant
sample_idx = X_valid_cat.index[0]

sample_row = X_valid_cat.loc[[sample_idx]]

# Model probability
sample_prob = cat_model.predict_proba(sample_row)[0, 1]

print("Applicant index:", sample_idx)
print(f"Predicted default probability: {sample_prob:.2%}")

In [ ]:
# SHAP explanation for this applicant
single_shap = explainer.shap_values(sample_row)

if isinstance(single_shap, list):
    single_shap = single_shap[1]

shap.plots.waterfall(
    shap.Explanation(
        values=single_shap[0],
        base_values=explainer.expected_value,
        data=sample_row.iloc[0],
        feature_names=sample_row.columns
    ),
    max_display=15
)

In [ ]:
# Convert model output from log-odds to probability
final_log_odds = explainer.expected_value + single_shap[0].sum()
final_probability = 1 / (1 + np.exp(-final_log_odds))

print(f"Predicted default probability: {final_probability:.2%}")

In [ ]:
# Top risk-increasing factors
contrib = pd.DataFrame({
    "feature": sample_row.columns,
    "value": sample_row.iloc[0].values,
    "shap": single_shap[0]
})

risk_increasing = contrib.sort_values("shap", ascending=False).head(10)
risk_reducing = contrib.sort_values("shap", ascending=True).head(10)

print("TOP RISK-INCREASING FACTORS")
display(risk_increasing)

print("TOP RISK-REDUCING FACTORS")
display(risk_reducing)

In [ ]:
cat_valid_prob = cat_model.predict_proba(X_valid_cat)[:, 1]

print(f"Mean predicted probability: {cat_valid_prob.mean():.2%}")
print(f"Actual default rate: {y_valid_fe.mean():.2%}")

In [ ]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix

thresholds = [0.03, 0.05, 0.08, 0.10, 0.15, 0.20]

results = []

for threshold in thresholds:
    pred = (cat_valid_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_valid_fe, pred
    ).ravel()

    results.append({
        "Threshold": threshold,
        "Flagged %": pred.mean(),
        "Precision": precision_score(y_valid_fe, pred, zero_division=0),
        "Recall": recall_score(y_valid_fe, pred, zero_division=0),
        "False Positives": fp,
        "False Negatives": fn
    })

threshold_results = pd.DataFrame(results)

display(
    threshold_results.style.format({
        "Threshold": "{:.0%}",
        "Flagged %": "{:.1%}",
        "Precision": "{:.1%}",
        "Recall": "{:.1%}"
    })
)

In [ ]:
risk_df = pd.DataFrame(index=X_valid_cat.index)

# Recover applicant IDs using the validation indices
risk_df["SK_ID_CURR"] = train_fe.loc[X_valid_cat.index, "SK_ID_CURR"]

risk_df["actual_default"] = y_valid_fe.values
risk_df["predicted_risk"] = cat_valid_prob

risk_df["risk_band"] = pd.cut(
    risk_df["predicted_risk"],
    bins=[-np.inf, 0.05, 0.10, 0.20, np.inf],
    labels=[
        "Lower Risk",
        "Moderate Risk",
        "Elevated Risk",
        "High Risk"
    ]
)

display(risk_df.head())

In [ ]:
risk_summary = (
    risk_df
    .groupby("risk_band", observed=True)
    .agg(
        applicants=("SK_ID_CURR", "count"),
        avg_predicted_risk=("predicted_risk", "mean"),
        actual_default_rate=("actual_default", "mean"),
        defaults=("actual_default", "sum")
    )
    .reset_index()
)

risk_summary["applicant_share"] = (
    risk_summary["applicants"] / len(risk_df)
)

display(
    risk_summary.style.format({
        "avg_predicted_risk": "{:.2%}",
        "actual_default_rate": "{:.2%}",
        "applicant_share": "{:.2%}"
    })
)

In [ ]:
train_fe.loc[X_valid_cat.index, "SK_ID_CURR"]

In [ ]:
# Check which important objects are available in memory

objects_to_check = [
    "train_fe",
    "test_fe",
    "X_train_fe",
    "X_valid_fe",
    "X_train_cat",
    "X_valid_cat",
    "y_valid_fe",
    "cat_model",
    "lgbm_fe",
    "xgb_model",
    "cat_valid_prob",
    "lgb_valid_prob",
    "xgb_valid_prob",
    "risk_df",
]

for name in objects_to_check:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {type(obj).__name__}", end="")

        if hasattr(obj, "shape"):
            print(f" | shape={obj.shape}")
        else:
            print()
    else:
        print(f"{name}: NOT FOUND")